<a href="https://colab.research.google.com/github/kemal1308/TugasBesarBigData-hm/blob/main/notebooks/01_data_cleaning_eda.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
!pip install pyspark

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
from pyspark.sql import SparkSession

# Membuat Spark Session
spark = SparkSession.builder \
    .appName("HM_Recommender_DataPrep") \
    .config("spark.driver.memory", "8g") \
    .getOrCreate()

# Tentukan path file di Google Drive (sesuaikan dengan nama folder Anda)
path_transactions = "/content/drive/MyDrive/Dataset_HM/transactions_train.csv"
path_articles = "/content/drive/MyDrive/Dataset_HM/articles.csv"
path_customers = "/content/drive/MyDrive/Dataset_HM/customers.csv"

# Memuat data CSV ke PySpark DataFrame
df_transactions = spark.read.csv(path_transactions, header=True, inferSchema=True)
df_articles = spark.read.csv(path_articles, header=True, inferSchema=True)
df_customers = spark.read.csv(path_customers, header=True, inferSchema=True)

# Menampilkan 5 baris pertama untuk memastikan data terbaca
df_transactions.show(5)

+----------+--------------------+----------+--------------------+----------------+
|     t_dat|         customer_id|article_id|               price|sales_channel_id|
+----------+--------------------+----------+--------------------+----------------+
|2018-09-20|000058a12d5b43e67...| 663713001|0.050830508474576264|               2|
|2018-09-20|000058a12d5b43e67...| 541518023| 0.03049152542372881|               2|
|2018-09-20|00007d2de826758b6...| 505221004| 0.01523728813559322|               2|
|2018-09-20|00007d2de826758b6...| 685687003|0.016932203389830508|               2|
|2018-09-20|00007d2de826758b6...| 685687004|0.016932203389830508|               2|
+----------+--------------------+----------+--------------------+----------------+
only showing top 5 rows


In [7]:
# 1. Menghapus baris yang memiliki nilai null pada kolom penting
df_transactions_clean = df_transactions.dropna(subset=["customer_id", "article_id"])

# 2. Menggabungkan (Join) data transaksi dengan data artikel untuk mendapatkan metadata produk
df_joined = df_transactions_clean.join(df_articles, on="article_id", how="left")

# (Opsional) Lakukan eksplorasi data (EDA) seperti menghitung total transaksi
print(f"Total transaksi bersih: {df_joined.count()}")

Total transaksi bersih: 31788324


In [8]:
# Menyimpan data hasil join ke format Parquet di Google Drive
output_path = "/content/drive/MyDrive/Dataset_HM/cleaned_data.parquet"
df_joined.write.mode("overwrite").parquet(output_path)